In [3]:
import numpy as np
import pdb

import control
import matplotlib.pyplot as plt
import torch
import torchsde
from torch import nn, profiler, Tensor, nan, vmap
import copy
import matplotlib.pyplot as plt

from tqdm import tqdm
import numpy as np
import pdb

import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..')) 
sys.path.append(project_root)  # Add folder_a to sys.path
sys.path.append(os.path.join(project_root, '../..', 'Networks'))  # Add folder_b to sys.path

from Networks.utils import get_gpu_memory, normal, ema_update,mass_append, update_ema, generate_system_canon, load_clean_state_dict, sample_linear, compute_gradnorm,save_model, loss_compute
from helpers import   HyperCoeffsLinearControlStochasticLQRImpl,  find_params
import psutil

from IPython.display import clear_output
from adabelief_pytorch import AdaBelief

is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

if not is_cuda:
    print("Warning: CUDA not available; falling back to CPU but this is likely to be very slow.")
import gc
path=os.path.abspath(os.getcwd())
gc.collect()
torch.cuda.empty_cache()

In [5]:
state_size=128

batch_size=8*256

weight=0.1

A,B=generate_system_canon(state_size)


In [9]:
learning_rate=0.001
alpha=0.999
gamma=0.9999
period=5000

iter=20000


global_perf=[]
global_eperf=[]
global_stperf=[]
global_grad=[]

At=torch.tensor(A).float().to(device) 
Bt=torch.tensor(B).float().to(device) 
coremat=(At.unsqueeze(0)).repeat(batch_size,1,1)
strategy="HNC"

t_total=1
t_size=int(32)
ts_exp = torch.linspace(0, t_total, t_size).float().to(device)

for layers in range(1,5):

    SML= HyperCoeffsLinearControlStochasticLQRImpl(At,Bt,batch_size, device,layers,strategy).to(device) 
    
    params =  find_params(SML)
    
    controller_optimizer = AdaBelief(params, lr=learning_rate, eps=1e-16, betas=(0.9,0.999), weight_decouple = True, rectify = True, amsgrad=False)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(controller_optimizer, gamma=gamma)
    
    
    SML.batch=batch_size
    
              
    score= 10000000
    
    ema_model = copy.deepcopy(SML)
    
                
    ema_loss, ema_ener, ema_std, enr_std = (0.0,) * 4

    eperf, stperf, eloss, stloss, emaperf, emastperf, emaener, emastener, gradnorms = ([] for _ in range(9))

   
    for j in tqdm(range(iter)):

        amat, x0= sample_linear(dim,batch_size, device)
       
        ini_enr = torch.zeros(batch_size, 1).to(device)
    
        ys_exp = torch.cat((x0,ini_enr),dim=1).to(device)
        
    
     
        coremat[:,-1,:]=amat
        SML.At= coremat
        SML.Bt= Bt
        SML.poly.amat=amat


        ys_tray = torchsde.sdeint_adjoint(SML, ys_exp, ts_exp,method='reversible_heun', dt=ts_exp[1]-ts_exp[0],
                             adjoint_method='adjoint_reversible_heun',)
        
                
        print(psutil.virtual_memory())
        
        lossvec, logtrick, stdvec, energy, stdener=loss_compute(ys_tray, state_size, weight) 

        print('total loss')
        print(lossvec)
 
        logtrick.backward()
        
        total_sq_norm=compute_gradnorm(SML)    

        print("grad norm")
        print(total_sq_norm)
    
     
        controller_optimizer.step()
        controller_optimizer.zero_grad(set_to_none=True)
        scheduler.step()
 
        update_ema(SML, ema_model, alpha)
        [ema_loss, ema_ener, enr_std, ema_std]=ema_update(j,alpha, lossvec,energy,stdener,stdvec,ema_loss,ema_ener,enr_std,ema_std)
       
        
        print('ema loss')
        print(ema_loss)
        print('ema_ener')
        print(ema_ener)
        
        [eperf, stperf, eloss, stloss, emaperf, emaener,emastperf,emastener, gradnorms]= mass_append(lossvec,stdvec,energy,stdener, batch_size, ema_loss, ema_ener, enr_std, ema_std, eperf, stperf, eloss, stloss, emaperf, emaener,emastperf,emastener,total_sq_norm, gradnorms)
       
        score= save_model(score, ema_loss, ema_model, SML, path,strategy,layers)
     
        if lossvec>500:
            break
        
        if j%200==0:
            gc.collect()
            clear_output(wait=True)
        del lossvec, stdvec 
    global_stperf.append(stperf)    
    global_perf.append(eperf)
    global_eperf.append(emaperf)
    global_grad.append(gradnorms)


  0%|          | 3/20000 [00:00<46:45,  7.13it/s]  

svmem(total=67025797120, available=48073576448, percent=28.3, used=17582333952, free=36659974144, active=18807857152, inactive=9125695488, buffers=1379438592, cached=11404050432, shared=623403008, slab=1134669824)
total loss
tensor(2.3979, device='cuda:0', grad_fn=<MeanBackward0>)
grad norm
0.004424169484818051
ema loss
2.4203666367530823
ema_ener
5.684362287138356e-05
model saved
svmem(total=67025797120, available=48073588736, percent=28.3, used=17582333952, free=36659974144, active=18807939072, inactive=9125707776, buffers=1379438592, cached=11404050432, shared=623403008, slab=1134669824)
total loss
tensor(2.4288, device='cuda:0', grad_fn=<MeanBackward0>)
grad norm
0.0046956850066235244
ema loss
2.4203750287375447
ema_ener
5.684447120773984e-05


  0%|          | 5/20000 [00:00<40:48,  8.17it/s]

svmem(total=67025797120, available=48072040448, percent=28.3, used=17583882240, free=36658425856, active=18808041472, inactive=9125707776, buffers=1379438592, cached=11404050432, shared=623403008, slab=1134669824)
total loss
tensor(2.3875, device='cuda:0', grad_fn=<MeanBackward0>)
grad norm
0.0036393030100985296
ema loss
2.420342174498907
ema_ener
5.684614139920114e-05
model saved
svmem(total=67025797120, available=48072040448, percent=28.3, used=17584005120, free=36658425856, active=18808074240, inactive=9125707776, buffers=1379438592, cached=11403927552, shared=623280128, slab=1134669824)
total loss
tensor(2.4340, device='cuda:0', grad_fn=<MeanBackward0>)
grad norm
0.004637111962911499
ema loss
2.4203558645261753
ema_ener
5.684646579644112e-05


  0%|          | 7/20000 [00:00<38:50,  8.58it/s]

...

  1%|          | 106/20000 [00:12<37:51,  8.76it/s]

grad norm
0.0015192211801350229
ema loss
2.4167966162906653
ema_ener
8.281633077375068e-05
model saved
svmem(total=67025797120, available=48057851904, percent=28.3, used=17589829632, free=36644171776, active=18820435968, inactive=9125773312, buffers=1379454976, cached=11412340736, shared=631640064, slab=1134669824)
total loss
tensor(2.3338, device='cuda:0', grad_fn=<MeanBackward0>)
grad norm
0.0014721313787109975
ema loss
2.4167135849427006
ema_ener
8.446925152459405e-05
model saved
svmem(total=67025797120, available=48057851904, percent=28.3, used=17589829632, free=36644171776, active=18820452352, inactive=9125773312, buffers=1379454976, cached=11412340736, shared=631640064, slab=1134669824)
total loss
tensor(2.3207, device='cuda:0', grad_fn=<MeanBackward0>)


  1%|          | 108/20000 [00:12<37:35,  8.82it/s]

grad norm
0.0015107963146836407
ema loss
2.4166175438698123
ema_ener
8.635357305009706e-05
model saved
svmem(total=67025797120, available=48057339904, percent=28.3, used=17590341632, free=36643659776, active=18820542464, inactive=9125773312, buffers=1379454976, cached=11412340736, shared=631640064, slab=1134669824)
total loss
tensor(2.3288, device='cuda:0', grad_fn=<MeanBackward0>)
grad norm
0.0016686781872212733
ema loss
2.416529747031356
ema_ener
8.848481121723794e-05
model saved
svmem(total=67025797120, available=48057339904, percent=28.3, used=17590341632, free=36643659776, active=18820567040, inactive=9125773312, buffers=1379454976, cached=11412340736, shared=631640064, slab=1134669824)
total loss
tensor(2.3487, device='cuda:0', grad_fn=<MeanBackward0>)


  1%|          | 108/20000 [00:12<38:45,  8.55it/s]

KeyboardInterrupt

